## hypothesis test: fake have less activated neurons than real

In [10]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon


REAL_CSV = "preds_of_64Neurons_denseLayer_test.csv"      # your real activations (64 cols + filenames)
FAKE_CSV= "[img+obj_labels_to_fakes]fake_activations(test).csv" 


ID_COL = "filenames"   # set to None if you don't have a filename/id column
ALPHA = 0.05
FIRE_FRAC = 0.80      # 80% of max_real


In [11]:
real_df = pd.read_csv(REAL_CSV)
fake_df = pd.read_csv(FAKE_CSV)
# fake_df["filenames"] = fake_df["filenames"].str.replace("fake_", "", regex=False)

if ID_COL is not None and ID_COL in real_df.columns and ID_COL in fake_df.columns:
    real_df = real_df.sort_values(ID_COL).reset_index(drop=True)
    fake_df = fake_df.sort_values(ID_COL).reset_index(drop=True)

    if not real_df[ID_COL].equals(fake_df[ID_COL]):
        raise ValueError("Real/Fake rows are not aligned by filename. Fix pairing first.")
else:
    # No ID column: assume rows already aligned in the same order
    if len(real_df) != len(fake_df):
        raise ValueError("Real/Fake have different number of rows; cannot assume pairing.")


In [12]:
exclude = {ID_COL} if ID_COL is not None else set()
common_cols = [c for c in real_df.columns if c in fake_df.columns and c not in exclude]
neuron_cols = [c for c in common_cols if pd.api.types.is_numeric_dtype(real_df[c])]

if len(neuron_cols) != 64:
    print(f"WARNING: expected 64 neuron columns, found {len(neuron_cols)}")

print("Using neuron columns:", len(neuron_cols))


Using neuron columns: 64


In [13]:
real_mat = real_df[neuron_cols].to_numpy(dtype=float)
fake_mat = fake_df[neuron_cols].to_numpy(dtype=float)

max_real = real_mat.max(axis=0)                 # shape: (64,)
thr = FIRE_FRAC * max_real                      # firing threshold per neuron

# Guard: if a neuron's max_real is 0, threshold is 0 (it never fires meaningfully)
# We'll handle it by requiring activation >= thr AND max_real > 0
valid_neuron = max_real > 0
print("Neurons with max_real>0:", valid_neuron.sum(), "/", len(valid_neuron))


Neurons with max_real>0: 64 / 64


In [14]:
# Step 1: zero filtering (most conservative)
keep = (real_mat > 0) & (fake_mat > 0)

# Step 2: firing condition (real-defined threshold)
real_fire = keep & (real_mat >= thr)
fake_fire = keep & (fake_mat >= thr)

# Step 3: count firing neurons per image
real_counts = real_fire.sum(axis=1)
fake_counts = fake_fire.sum(axis=1)

# real_fire = (real_mat[:, valid_neuron] >= thr[valid_neuron])
# fake_fire = (fake_mat[:, valid_neuron] >= thr[valid_neuron])

# real_counts = real_fire.sum(axis=1)   # firing neuron count per real image
# fake_counts = fake_fire.sum(axis=1)   # firing neuron count per fake image

print("Example counts (first 10):")
print("real:", real_counts[:10])
print("fake:", fake_counts[:10])


Example counts (first 10):
real: [2 0 8 0 0 2 1 0 0 1]
fake: [0 0 0 0 0 2 6 0 1 1]


In [15]:
diff = real_counts - fake_counts

# Wilcoxon ignores zeros in diff; make sure we have enough nonzero pairs
nz = diff != 0
if nz.sum() < 10:
    raise ValueError(f"Too few nonzero paired differences: {nz.sum()}")

stat, p = wilcoxon(diff[nz], alternative="greater", zero_method="wilcox")

wins = np.sum(diff[nz] > 0)
losses = np.sum(diff[nz] < 0)
r_rb = (wins - losses) / (wins + losses)  # rank-biserial (sign-based)

print("=== H1: fake has fewer activated neurons than real ===")
print(f"Images paired              : {len(diff)}")
print(f"Nonzero diffs used          : {nz.sum()}")
print(f"Mean firing count (real)    : {real_counts.mean():.3f}")
print(f"Mean firing count (fake)    : {fake_counts.mean():.3f}")
print(f"Median(real-fake)           : {np.median(diff[nz]):.3f}")
print(f"Mean(real-fake)           : {np.mean(diff[nz]):.3f}")
print(f"Prop(real>fake)             : {(diff[nz] > 0).mean():.3f}")
print(f"Wilcoxon stat               : {stat:.6e}")
print(f"p-value (one-sided, real>fake): {p:.6e}")
print(f"rank-biserial r_rb          : {r_rb:.3f}  (positive favors real)")
print("Decision:", "REJECT H0" if p < ALPHA else "fail to reject H0", f"at alpha={ALPHA}")


=== H1: fake has fewer activated neurons than real ===
Images paired              : 793
Nonzero diffs used          : 348
Mean firing count (real)    : 0.776
Mean firing count (fake)    : 0.430
Median(real-fake)           : 1.000
Mean(real-fake)           : 0.787
Prop(real>fake)             : 0.681
Wilcoxon stat               : 4.293300e+04
p-value (one-sided, real>fake): 2.939991e-12
rank-biserial r_rb          : 0.362  (positive favors real)
Decision: REJECT H0 at alpha=0.05


In [79]:
## for only confirmed neurons

import re
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

REAL_CSV = "preds_of_64Neurons_denseLayer_test.csv"      # your real activations (64 cols + filenames)
FAKE_CSV= "[img+obj_labels_to_fakes]fake_activations(test).csv" 
CONFIRMED_CSV = "verification_summary_1 (test).csv"   # has column: neuron_id

ID_COL = "filenames"   # set to None if you don't have this column
ALPHA = 0.01


# confirmed_df = pd.read_csv(CONFIRMED_CSV)
# confirmed_ids = set(confirmed_df["neuron_id"].astype(int))

# print("Confirmed neurons:", sorted(confirmed_ids))
# print("Count:", len(confirmed_ids))


In [80]:
real_df = pd.read_csv(REAL_CSV)
fake_df = pd.read_csv(FAKE_CSV)
# fake_df["filenames"] = fake_df["filenames"].str.replace("fake_", "", regex=False)

if ID_COL is not None and ID_COL in real_df.columns and ID_COL in fake_df.columns:
    real_df = real_df.sort_values(ID_COL).reset_index(drop=True)
    fake_df = fake_df.sort_values(ID_COL).reset_index(drop=True)
    if not real_df[ID_COL].equals(fake_df[ID_COL]):
        raise ValueError("Real/Fake rows do not align by filename. Fix pairing first.")
else:
    if len(real_df) != len(fake_df):
        raise ValueError("Real/Fake have different row counts; cannot assume pairing.")


In [81]:
exclude = {ID_COL} if ID_COL is not None else set()
common_cols = [c for c in real_df.columns if c in fake_df.columns and c not in exclude]
neuron_cols = [c for c in common_cols if pd.api.types.is_numeric_dtype(real_df[c])]

if len(neuron_cols) == 0:
    raise ValueError("No numeric neuron columns found in both CSVs.")

print("Numeric activation columns found:", len(neuron_cols))


Numeric activation columns found: 64


In [82]:
conf_df = pd.read_csv(CONFIRMED_CSV)

def extract_int(x):
    m = re.search(r"\d+", str(x))
    return int(m.group()) if m else None

confirmed_cols = []

# Case A: confirmed file contains actual activation column names
for cand in ["colname", "column", "neuron_col", "neuron_name", "neuron"]:
    if cand in conf_df.columns:
        vals = conf_df[cand].dropna().astype(str).tolist()
        confirmed_cols = [v for v in vals if v in neuron_cols]
        if confirmed_cols:
            break

# Case B: confirmed file contains neuron ids (0..63)
if not confirmed_cols:
    for cand in ["neuron_id", "id", "neuron"]:
        if cand in conf_df.columns:
            ids = [extract_int(v) for v in conf_df[cand].dropna().tolist()]
            ids = [i for i in ids if i is not None]
            ids = set(ids)

            # Map ids to columns by matching integer found in column name
            def col_id(col): 
                return extract_int(col)

            confirmed_cols = [c for c in neuron_cols if col_id(c) in ids]
            if confirmed_cols:
                break

if not confirmed_cols:
    raise ValueError(
        "Could not map confirmed neurons to activation columns. "
        "Ensure confirmed_neurons.csv has column names or neuron ids."
    )

print("Confirmed neuron columns:", len(confirmed_cols))
print("Example:", confirmed_cols[:10])


Confirmed neuron columns: 25
Example: ['0', '7', '9', '11', '12', '16', '19', '20', '27', '28']


In [83]:
import numpy as np
from scipy.stats import wilcoxon

R = real_df[confirmed_cols].to_numpy(dtype=float)  # (N_images, N_confirmed)
F = fake_df[confirmed_cols].to_numpy(dtype=float)

max_real = R.max(axis=0)
thr = 0.8 * max_real

valid = max_real > 0
R = R[:, valid]
F = F[:, valid]
thr = thr[valid]

print("Confirmed neurons used (max_real>0):", thr.size)

keep = (R > 0) & (F > 0)

print("Total positions:", R.size)
print("Kept nonzero positions:", keep.sum())


real_active = keep & (R >= thr)
fake_active = keep & (F >= thr)

real_counts = real_active.sum(axis=1)   # per image
fake_counts = fake_active.sum(axis=1)

print("Mean activated (real):", real_counts.mean())
print("Mean activated (fake):", fake_counts.mean())


Confirmed neurons used (max_real>0): 25
Total positions: 19825
Kept nonzero positions: 6523
Mean activated (real): 0.31399747793190413
Mean activated (fake): 0.15510718789407313


In [84]:
diff = real_counts - fake_counts
nz = diff != 0

if nz.sum() < 10:
    raise ValueError(f"Too few nonzero count differences for Wilcoxon: {nz.sum()}")

stat, p = wilcoxon(diff[nz], alternative="greater", zero_method="wilcox")

wins = np.sum(diff[nz] > 0)
losses = np.sum(diff[nz] < 0)
r_rb = (wins - losses) / (wins + losses)

print("\n=== Zero-filtered + 80% threshold activation count test (confirmed neurons) ===")
print("keep: (real>0 AND fake>0)")
print("activated: (>= 0.8 * max_real) within kept positions")
print(f"Images paired                 : {len(diff)}")
print(f"Nonzero diffs used            : {nz.sum()}")
print(f"Mean #activated (real)        : {real_counts.mean():.3f}")
print(f"Mean #activated (fake)        : {fake_counts.mean():.3f}")
print(f"Median(real-fake)             : {np.median(diff[nz]):.3f}")
print(f"Prop(real>fake)               : {(diff[nz] > 0).mean():.3f}")
print(f"Wilcoxon stat                 : {stat:.6e}")
print(f"p-value (one-sided real>fake) : {p:.6e}")
print(f"rank-biserial r_rb            : {r_rb:.3f}")
print("Decision:", "REJECT H0" if p < ALPHA else "fail to reject H0", f"at alpha={ALPHA}")



=== Zero-filtered + 80% threshold activation count test (confirmed neurons) ===
keep: (real>0 AND fake>0)
activated: (>= 0.8 * max_real) within kept positions
Images paired                 : 793
Nonzero diffs used            : 189
Mean #activated (real)        : 0.314
Mean #activated (fake)        : 0.155
Median(real-fake)             : 1.000
Prop(real>fake)               : 0.709
Wilcoxon stat                 : 1.294700e+04
p-value (one-sided real>fake) : 1.724670e-08
rank-biserial r_rb            : 0.418
Decision: REJECT H0 at alpha=0.01
